## Flood Risk Zone Mapping

#### Bristol Testing

In [1]:
import geopandas as gpd
flood_zones = gpd.read_file("Flood_Map_for_Planning_Flood_Zones.gpkg")

In [2]:
from shapely.geometry import box
bbox = box(357000, 169000, 361000, 173000)  # EPSG:27700 coords
clipped = flood_zones[flood_zones.intersects(bbox)]

In [3]:
clipped.to_file("bristol_floodzones.geojson", driver="GeoJSON")

In [4]:
import folium
m = folium.Map(location=[51.455, -2.587], zoom_start=12)
folium.GeoJson("bristol_floodzones.geojson").add_to(m)
m.save("bristol_map.html")

In [6]:

import json
from folium import plugins

# Load clipped GeoJSON
with open("bristol_floodzones.geojson", "r", encoding="utf-8") as f:
    data = json.load(f)

# Create map centered on Bristol
m = folium.Map(location=[51.455, -2.587], zoom_start=12, tiles="CartoDB positron")

# Define style function based on flood zone
def style_function(feature):
    zone = feature["properties"].get("flood_zone", "")
    color = "#1f78b4" if zone == "FZ2" else "#bd0026" if zone == "FZ3" else "#8c8c8c"
    return {
        "fillColor": color,
        "color": color,
        "weight": 0.5,
        "fillOpacity": 0.4,
        "opacity": 0.6,
    }

# Tooltip fields (adjust if needed)
tooltip = folium.GeoJsonTooltip(
    fields=["flood_zone", "origin", "flood_source"],
    aliases=["Flood Zone", "Origin", "Source"],
    localize=True,
    sticky=False,
)

# Add flood zones layer
folium.GeoJson(
    data,
    name="Flood Zones 2 & 3",
    style_function=style_function,
    tooltip=tooltip,
).add_to(m)

# Add controls
folium.LayerControl(collapsed=False).add_to(m)
plugins.Fullscreen(position="topright").add_to(m)
plugins.MeasureControl(position="topright").add_to(m)

# Save map
m.save("bristol_map.html")
print("[COMPLETE] Map saved to bristol_map.html")

[COMPLETE] Map saved to bristol_map.html


In [7]:
print(data["features"][0]["geometry"]["type"])
print(data["features"][0]["geometry"]["coordinates"][:2])

MultiPolygon
[[[[357415.0001001358, 169290.0001001358], [357415.0001001358, 169292.5001001358], [357417.5001001358, 169292.5001001358], [357417.5001001358, 169290.0001001358], [357415.0001001358, 169290.0001001358]]]]


In [8]:
print(len(data["features"]))

2351


In [9]:
import geopandas as gpd
import folium
import json
from shapely.geometry import box

# Step 1: Load GeoPackage
print("Loading GeoPackage...")
flood_zones = gpd.read_file("Flood_Map_for_Planning_Flood_Zones.gpkg")
print(f"✓ Loaded {len(flood_zones)} features.")

# Step 2: Clip to Bristol bounding box (EPSG:27700)
print("Clipping to Bristol bounding box...")
bbox = box(357000, 169000, 361000, 173000)
clipped = flood_zones[flood_zones.intersects(bbox)]
print(f"✓ Clipped to {len(clipped)} features.")

# Step 3: Reproject to WGS84 for Folium
print("Reprojecting to WGS84...")
clipped = clipped.to_crs(epsg=4326)
print("✓ Reprojection complete.")

# Step 4: Export to GeoJSON
print("Exporting to GeoJSON...")
clipped.to_file("bristol_floodzones.geojson", driver="GeoJSON")
print("✓ Exported to bristol_floodzones.geojson")

# Step 5: Load GeoJSON for Folium
print("Loading GeoJSON for Folium...")
with open("bristol_floodzones.geojson", "r", encoding="utf-8") as f:
    data = json.load(f)

# Step 6: Create Folium map
print("Creating Folium map...")
m = folium.Map(location=[51.455, -2.587], zoom_start=12, tiles="CartoDB positron")

# Step 7: Add flood zone overlay
folium.GeoJson(
    data,
    name="Flood Zones",
    tooltip=folium.GeoJsonTooltip(
        fields=["flood_zone"],
        aliases=["Flood Zone"],
        sticky=False
    )
).add_to(m)

# Step 8: Add layer control and save
folium.LayerControl().add_to(m)
m.save("bristol_floodzones_map.html")
print("✓ Map saved to bristol_floodzones_map.html")

Loading GeoPackage...
✓ Loaded 3536992 features.
Clipping to Bristol bounding box...
✓ Clipped to 2351 features.
Reprojecting to WGS84...
✓ Reprojection complete.
Exporting to GeoJSON...
✓ Exported to bristol_floodzones.geojson
Loading GeoJSON for Folium...
Creating Folium map...
✓ Map saved to bristol_floodzones_map.html


In [10]:
import json
import folium
from folium import plugins

# Load the clipped GeoJSON you already exported
with open("bristol_floodzones.geojson", "r", encoding="utf-8") as f:
    data = json.load(f)

# Create map centered on Bristol
m = folium.Map(location=[51.455, -2.587], zoom_start=12, tiles="CartoDB positron")

# Style function for Flood Zone 2 vs 3
def style_function(feature):
    zone = feature["properties"].get("flood_zone", "")
    if zone == "FZ2":
        color = "#1f78b4"  # blue
    elif zone == "FZ3":
        color = "#bd0026"  # red
    else:
        color = "#8c8c8c"  # grey fallback
    return {
        "fillColor": color,
        "color": color,
        "weight": 0.5,
        "fillOpacity": 0.4,
        "opacity": 0.6,
    }

# Add flood zones overlay with tooltip
folium.GeoJson(
    data,
    name="Flood Zones 2 & 3",
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["flood_zone"],
        aliases=["Flood Zone"],
        sticky=False
    )
).add_to(m)

# Add controls and save
folium.LayerControl(collapsed=False).add_to(m)
plugins.Fullscreen(position="topright").add_to(m)
plugins.MeasureControl(position="topright").add_to(m)

m.save("bristol_floodzones_map.html")
print("✓ Map saved to bristol_floodzones_map.html")

✓ Map saved to bristol_floodzones_map.html


#### Calderdale Plotting

In [1]:
import geopandas as gpd
from shapely.geometry import box
from pyproj import Transformer
import folium
import json

# ----- 1. Define Calderdale bounds in WGS84 -----
min_lon, max_lon = -2.25, -1.75
min_lat, max_lat = 53.55, 53.85

# ----- 2. Transform bounds to EPSG:27700 -----
transformer = Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True)

min_x, min_y = transformer.transform(min_lon, min_lat)
max_x, max_y = transformer.transform(max_lon, max_lat)

print(f"Calderdale bbox in EPSG:27700:")
print(f"  min_x = {min_x:.1f}, min_y = {min_y:.1f}")
print(f"  max_x = {max_x:.1f}, max_y = {max_y:.1f}")

# ----- 3. Load flood zones GeoPackage -----
print("Loading GeoPackage...")
flood_zones = gpd.read_file("Flood_Map_for_Planning_Flood_Zones.gpkg")
print(f"✓ Loaded {len(flood_zones)} features.")

# ----- 4. Clip to Calderdale in EPSG:27700 -----
print("Clipping to Calderdale bounding box...")
calderdale_bbox = box(min_x, min_y, max_x, max_y)
clipped = flood_zones[flood_zones.intersects(calderdale_bbox)]
print(f"✓ Clipped to {len(clipped)} features.")

# Optional: filter only FZ2 and FZ3
if "flood_zone" in clipped.columns:
    clipped = clipped[clipped["flood_zone"].isin(["FZ2", "FZ3"])]
    print(f"✓ After filtering FZ2/FZ3: {len(clipped)} features.")

# ----- 5. Reproject to WGS84 and save GeoJSON -----
print("Reprojecting to WGS84...")
clipped_wgs84 = clipped.to_crs(epsg=4326)

geojson_path = "calderdale_floodzones_2_3.geojson"
print(f"Exporting to GeoJSON: {geojson_path}...")
clipped_wgs84.to_file(geojson_path, driver="GeoJSON")
print("✓ GeoJSON exported.")

# ----- 6. Build Folium map with coloured overlays -----
print("Creating Folium map...")
with open(geojson_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Center roughly on Calderdale
center_lat = (min_lat + max_lat) / 2
center_lon = (min_lon + max_lon) / 2

m = folium.Map(location=[center_lat, center_lon],
               zoom_start=11,
               tiles="CartoDB positron")

def style_function(feature):
    zone = feature["properties"].get("flood_zone", "")
    if zone == "FZ2":
        color = "#1f78b4"  # blue
    elif zone == "FZ3":
        color = "#bd0026"  # red
    else:
        color = "#8c8c8c"  # grey fallback
    return {
        "fillColor": color,
        "color": color,
        "weight": 0.5,
        "fillOpacity": 0.4,
        "opacity": 0.6,
    }

tooltip = folium.GeoJsonTooltip(
    fields=["flood_zone"],
    aliases=["Flood Zone"],
    sticky=False,
)

folium.GeoJson(
    data,
    name="Calderdale Flood Zones 2 & 3",
    style_function=style_function,
    tooltip=tooltip,
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save("calderdale_floodzones_2_3.html")
print("✓ Map saved to calderdale_floodzones_2_3.html")

Calderdale bbox in EPSG:27700:
  min_x = 383533.2, min_y = 405957.0
  max_x = 416544.5, max_y = 439334.4
Loading GeoPackage...
✓ Loaded 3536992 features.
Clipping to Calderdale bounding box...
✓ Clipped to 6284 features.
✓ After filtering FZ2/FZ3: 6284 features.
Reprojecting to WGS84...
Exporting to GeoJSON: calderdale_floodzones_2_3.geojson...
✓ GeoJSON exported.
Creating Folium map...
✓ Map saved to calderdale_floodzones_2_3.html


#### Lincolnshire Plotting

In [2]:
import json
import folium
from pyproj import Transformer
from shapely.geometry import box
import geopandas as gpd

# ----- 1. Define Lincolnshire bounds in WGS84 -----
min_lon, max_lon = -0.90, 0.40
min_lat, max_lat = 52.70, 53.60

# ----- 2. Transform bounds to EPSG:27700 -----
transformer = Transformer.from_crs("EPSG:4326", "EPSG:27700", always_xy=True)

min_x, min_y = transformer.transform(min_lon, min_lat)
max_x, max_y = transformer.transform(max_lon, max_lat)

print("Lincolnshire bounding box (EPSG:27700):")
print(f"  min_x = {min_x:.1f}, min_y = {min_y:.1f}")
print(f"  max_x = {max_x:.1f}, max_y = {max_y:.1f}")

# ----- 3. Load flood zones GeoPackage -----
print("Loading GeoPackage...")
flood_zones = gpd.read_file("Flood_Map_for_Planning_Flood_Zones.gpkg")
print(f"✓ Loaded {len(flood_zones)} features.")

# ----- 4. Clip to Lincolnshire -----
print("Clipping to Lincolnshire bounding box...")
lincolnshire_bbox = box(min_x, min_y, max_x, max_y)
clipped = flood_zones[flood_zones.intersects(lincolnshire_bbox)]
print(f"✓ Clipped to {len(clipped)} features.")

# Optional: filter only FZ2 and FZ3
if "flood_zone" in clipped.columns:
    clipped = clipped[clipped["flood_zone"].isin(["FZ2", "FZ3"])]
    print(f"✓ After filtering FZ2/FZ3: {len(clipped)} features.")

# ----- 5. Reproject to WGS84 and save GeoJSON -----
print("Reprojecting to WGS84...")
clipped_wgs84 = clipped.to_crs(epsg=4326)

geojson_path = "lincolnshire_floodzones_2_3.geojson"
print(f"Exporting to GeoJSON: {geojson_path}...")
clipped_wgs84.to_file(geojson_path, driver="GeoJSON")
print("✓ GeoJSON exported.")

# ----- 6. Build Folium map with coloured overlays -----
print("Creating Folium map...")
with open(geojson_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Center on Lincolnshire
center_lat = (min_lat + max_lat) / 2
center_lon = (min_lon + max_lon) / 2

m = folium.Map(location=[center_lat, center_lon],
               zoom_start=9,
               tiles="CartoDB positron")

def style_function(feature):
    zone = feature["properties"].get("flood_zone", "")
    if zone == "FZ2":
        color = "#1f78b4"  # blue
    elif zone == "FZ3":
        color = "#bd0026"  # red
    else:
        color = "#8c8c8c"  # grey fallback
    return {
        "fillColor": color,
        "color": color,
        "weight": 0.5,
        "fillOpacity": 0.4,
        "opacity": 0.6,
    }

tooltip = folium.GeoJsonTooltip(
    fields=["flood_zone"],
    aliases=["Flood Zone"],
    sticky=False,
)

folium.GeoJson(
    data,
    name="Lincolnshire Flood Zones 2 & 3",
    style_function=style_function,
    tooltip=tooltip,
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save("lincolnshire_floodzones_2_3.html")
print("✓ Map saved to lincolnshire_floodzones_2_3.html")

Lincolnshire bounding box (EPSG:27700):
  min_x = 474429.7, min_y = 311934.6
  max_x = 558912.1, max_y = 414167.1
Loading GeoPackage...
✓ Loaded 3536992 features.
Clipping to Lincolnshire bounding box...
✓ Clipped to 122781 features.
✓ After filtering FZ2/FZ3: 122781 features.
Reprojecting to WGS84...
Exporting to GeoJSON: lincolnshire_floodzones_2_3.geojson...
✓ GeoJSON exported.
Creating Folium map...
✓ Map saved to lincolnshire_floodzones_2_3.html


#### Lincolnshire Simplified Plotting (Original plot too big)

In [6]:
import geopandas as gpd

print("Simplifying geometry while preserving attributes...")

simplified = clipped_wgs84.copy()
simplified["geometry"] = simplified.geometry.simplify(
    tolerance=0.0001, 
    preserve_topology=True
)

print("✓ Simplification complete.")
print("Columns preserved:", simplified.columns)

output_path = "lincolnshire_floodzones_2_3_simplified.geojson"
simplified.to_file(output_path, driver="GeoJSON")

print(f"✓ Saved to {output_path}")

Simplifying geometry while preserving attributes...
✓ Simplification complete.
Columns preserved: Index(['origin', 'flood_zone', 'flood_source', 'shape_length', 'shape_area',
       'geometry'],
      dtype='object')
✓ Saved to lincolnshire_floodzones_2_3_simplified.geojson


In [7]:

# Load your simplified GeoJSON
with open("lincolnshire_floodzones_2_3_simplified.geojson", "r", encoding="utf-8") as f:
    data = json.load(f)

# Basic structure
print("Top-level keys:", data.keys())

# Number of features
features = data.get("features", [])
print("Number of features:", len(features))

# Inspect first feature
if len(features) > 0:
    props = features[0].get("properties", {})
    print("\nProperties in first feature:")
    for k, v in props.items():
        print(f"  {k}: {v}")

    print("\nList of property keys:")
    print(list(props.keys()))
else:
    print("No features found in the GeoJSON.")

Top-level keys: dict_keys(['type', 'name', 'crs', 'features'])
Number of features: 122781

Properties in first feature:
  origin: modelled
  flood_zone: FZ3
  flood_source: river and sea
  shape_length: None
  shape_area: None

List of property keys:
['origin', 'flood_zone', 'flood_source', 'shape_length', 'shape_area']


In [8]:
import json
import folium

with open("lincolnshire_floodzones_2_3_simplified.geojson", "r", encoding="utf-8") as f:
    data = json.load(f)

m = folium.Map(location=[53.15, -0.25], zoom_start=9, tiles="CartoDB positron")

def style_function(feature):
    zone = feature["properties"].get("flood_zone", "")
    if zone == "FZ2":
        color = "#1f78b4"
    elif zone == "FZ3":
        color = "#bd0026"
    else:
        color = "#8c8c8c"
    return {
        "fillColor": color,
        "color": color,
        "weight": 0.5,
        "fillOpacity": 0.4,
        "opacity": 0.6,
    }

folium.GeoJson(
    data,
    name="Lincolnshire Flood Zones (Simplified)",
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=["flood_zone"], aliases=["Flood Zone"])
).add_to(m)

folium.LayerControl().add_to(m)
m.save("lincolnshire_floodzones_2_3_simplified.html")

print("✓ Simplified map saved.")

✓ Simplified map saved.
